# Online Shoppers Intention Analysis


## Part 1: 数据加载与探索性分析

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 基础显示设置
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.unicode_minus"] = False

# 尝试启用常见中文字体
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]

### 数据加载与基础检查
读取数据后查看数据规模、样例、字段类型、统计摘要与缺失值情况。

In [ ]:
data_path = "data/online_shoppers_intention.csv"
df = pd.read_csv(data_path)

print(f"数据形状: {df.shape}")
display(df.head())

print("\nDataFrame.info():")
df.info()

print("\n数值统计摘要 (describe):")
display(df.describe())

print("\n包含类别变量在内的统计摘要 (describe include='all'):")
display(df.describe(include="all"))

In [ ]:
# 缺失值统计（数量 + 占比）
missing_count = df.isnull().sum()
missing_ratio = (missing_count / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_ratio_percent": missing_ratio
}).sort_values("missing_ratio_percent", ascending=False)

display(missing_summary)

if missing_summary["missing_count"].sum() == 0:
    print("未发现缺失值。")
else:
    print("存在缺失值，请在后续预处理中处理。")

### 目标变量分布分析
统计 `Revenue` 的类别数量与占比，并使用柱状图 + 饼图展示类别不平衡。

In [ ]:
if "Revenue" not in df.columns:
    raise KeyError("未找到目标变量列 'Revenue'。")

revenue_counts = df["Revenue"].value_counts(dropna=False)
revenue_ratio = df["Revenue"].value_counts(normalize=True, dropna=False).mul(100).round(2)

print("Revenue 类别计数:")
display(revenue_counts.to_frame(name="count"))

print("Revenue 类别占比(%):")
display(revenue_ratio.to_frame(name="ratio_percent"))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x="Revenue", data=df, ax=axes[0], palette="Set2")
axes[0].set_title("Revenue 类别分布（柱状图）")
axes[0].set_xlabel("Revenue")
axes[0].set_ylabel("Count")

axes[1].pie(
    revenue_counts.values,
    labels=revenue_counts.index.astype(str),
    autopct="%.1f%%",
    startangle=90,
    colors=sns.color_palette("Set2", n_colors=len(revenue_counts))
)
axes[1].set_title("Revenue 类别分布（饼图）")

plt.tight_layout()
plt.show()

### 数值特征分布可视化
绘制数值特征的直方图与箱线图，观察分布形态与异常值。

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"数值特征数量: {len(num_cols)}")
print(num_cols)

# 直方图
_ = df[num_cols].hist(bins=30, figsize=(18, 12), edgecolor="black")
plt.suptitle("数值特征直方图", y=1.02)
plt.tight_layout()
plt.show()

# 箱线图
plt.figure(figsize=(14, max(6, len(num_cols) * 0.35)))
sns.boxplot(data=df[num_cols], orient="h", palette="Set3")
plt.title("数值特征箱线图")
plt.xlabel("Value")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

### 类别特征频率统计
统计 `Month`、`VisitorType`、`Weekend` 的频数与占比，并绘制柱状图。

In [ ]:
cat_cols = ["Month", "VisitorType", "Weekend"]
existing_cat_cols = [c for c in cat_cols if c in df.columns]
missing_cat_cols = [c for c in cat_cols if c not in df.columns]

if missing_cat_cols:
    print(f"以下类别列不存在，将跳过: {missing_cat_cols}")

for col in existing_cat_cols:
    print(f"\n=== {col} 频数统计 ===")
    counts = df[col].value_counts(dropna=False)
    ratio = df[col].value_counts(normalize=True, dropna=False).mul(100).round(2)
    summary = pd.DataFrame({"count": counts, "ratio_percent": ratio})
    display(summary)

if existing_cat_cols:
    n = len(existing_cat_cols)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, col in zip(axes, existing_cat_cols):
        order = df[col].value_counts(dropna=False).index
        sns.countplot(data=df, x=col, order=order, ax=ax, palette="pastel")
        ax.set_title(f"{col} 频数分布")
        ax.set_xlabel(col)
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

### 相关性热力图
计算数值特征的相关系数矩阵并绘制热力图。

In [ ]:
corr = df[num_cols].corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr,
    cmap="RdBu_r",
    center=0,
    square=False,
    cbar_kws={"shrink": 0.8}
)
plt.title("数值特征相关性热力图")
plt.tight_layout()
plt.show()